#### Approach 1

In [1]:
import warnings
warnings.filterwarnings('ignore')
from langchain_community.tools import tool

In [2]:
# Step 1: Create custom function
def add(a, b):
    """Performs addition of two numbers"""
    return a + b

In [3]:
# Step 2: Add type hints
def add(a: float = 0, b: float = 0) -> float:
    """Performs addition of two numbers"""
    return a + b

In [4]:
# Step 3: Add tool decorator
@tool
def add(a: float = 0.0, b: float = 0.0) -> float:
    """Performs addition of two numbers"""
    return a + b

# Our tool now becomes a runnable and LLMs can now interact with it

In [5]:
result = add.invoke(input = {
    "a": float(2),
    "b": float(17.2)
})
print(result)

19.2


In [6]:
result = add.invoke(input = {})
print(result)

0.0


In [7]:
# This tool has some properties
print(add.name)
print(add.description)
print(add.args)

add
Performs addition of two numbers
{'a': {'default': 0.0, 'title': 'A', 'type': 'number'}, 'b': {'default': 0.0, 'title': 'B', 'type': 'number'}}


In [8]:
# How LLM sees this tool
import json
data = add.args_schema.model_json_schema()
formatted = json.dumps(data, indent=4)
print(formatted)

{
    "description": "Performs addition of two numbers",
    "properties": {
        "a": {
            "default": 0.0,
            "title": "A",
            "type": "number"
        },
        "b": {
            "default": 0.0,
            "title": "B",
            "type": "number"
        }
    },
    "title": "add",
    "type": "object"
}


#### Approach 2

In [9]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field

In [10]:
class MultiplyInput(BaseModel):
    a: float = Field(description = "First input", required = True)
    b: float = Field(description = "Second input", required = True)

In [11]:
def multiply(a: float, b: float) -> float:
    """Performs multiplication of two numbers"""
    return a * b

In [12]:
multiply_tool = StructuredTool.from_function(
    func = multiply,
    name = "multiply_tool",
    description = "Performs multiplication of two numbers",
    args_schema = MultiplyInput
)

In [13]:
print(multiply_tool.invoke({
    "a": 27.1,
    "b": 24.227
}))

656.5517000000001


In [14]:
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

multiply_tool
Performs multiplication of two numbers
{'a': {'description': 'First input', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second input', 'title': 'B', 'type': 'number'}}


In [15]:
import json
data = multiply_tool.args_schema.model_json_schema()
formatted = json.dumps(data, indent=4)
print(formatted)

{
    "properties": {
        "a": {
            "description": "First input",
            "required": true,
            "title": "A",
            "type": "number"
        },
        "b": {
            "description": "Second input",
            "required": true,
            "title": "B",
            "type": "number"
        }
    },
    "required": [
        "a",
        "b"
    ],
    "title": "MultiplyInput",
    "type": "object"
}


#### Approach 3

In [16]:
from langchain_community.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

In [17]:
class MultiplyInput(BaseModel):
    a: float = Field(description = "First input", required = True)
    b: float = Field(description = "Second input", required = True)

In [18]:
class MultiplyTool(BaseTool):
    name: str = "multiply_tool"
    description: str = "Performs multiplication of two numbers"
    
    args_schema: Type[BaseModel] = MultiplyInput
    def _run(self, a: float, b: float) -> float:
        return a * b

In [19]:
multiply_t = MultiplyTool()
print(multiply_t.invoke(input = {
    "a": 10.10,
    "b": 100.00
}))

1010.0


In [20]:
print(multiply_t.name)
print(multiply_t.description)
print(multiply_t.args)

multiply_tool
Performs multiplication of two numbers
{'a': {'description': 'First input', 'title': 'A', 'type': 'number'}, 'b': {'description': 'Second input', 'title': 'B', 'type': 'number'}}


In [21]:
import json
data = multiply_t.args_schema.model_json_schema()
formatted = json.dumps(data, indent=4)
print(formatted)

{
    "properties": {
        "a": {
            "description": "First input",
            "required": true,
            "title": "A",
            "type": "number"
        },
        "b": {
            "description": "Second input",
            "required": true,
            "title": "B",
            "type": "number"
        }
    },
    "required": [
        "a",
        "b"
    ],
    "title": "MultiplyInput",
    "type": "object"
}


#### ToolKit - Multiple tools

In [22]:
from langchain_core.tools import tool
@tool
def add(a: float, b: float) -> float:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a"""
    return a - b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b"""
    return a / b

In [24]:
class BasicMathToolkit:
    def get_tools(self):
        return [add, multiply, subtract, divide]

toolkit = BasicMathToolkit()
tools = toolkit.get_tools()
for tool in tools:
    print(f'{tool.name} ======> {tool.description}')

add ======> Add two numbers
multiply ======> Multiply two numbers
subtract ======> Subtract b from a
divide ======> Divide a by b
